# **1. Construção do banco de dados: C. auris**

Este notebook faz parte de uma série de notebooks com atribuições próprias que vão alicerçar a construção de uma CLI de vigilância genômica. Suas atribuições fundamentais são:

- Interação com a api do NCBI para requisitar todos os dados FASTA disponíveis para o taxon *C. auris*;
- Organização das pastas para armazenamento dos dados brutos e processados;
- Executar um *Quality Control* básico:
    - size_mb muito baixo (esperado ~12mb para *C. auris*);
    - assembly marcado como partial;
    - assembly marcado como atypical;
    - arquivo .fna ausente;
    - metadado sem caminho válido.
- Mover genomas filtrados/excluidos para uma pasta específica;
- Registrar em CSV o motivo da exclusão de cada genoma;
- Salvar metadados limpos em formato tabular;
- Preparar a pasta-base de genomas válidos para as próximas etapas do pipeline.

Neste notebook, construiremos ainda o primeiro branch para armazenamento do banco de dados, seguindo a seguinte estruturação:

```text
data/
└── raw/
    ├── v1/
    │   ├── genomes_all/
    │   ├── genomes_excluded/
    │   ├── metadata/
    │   │   └── assembly_data_report.jsonl
    │   ├── sample_metadata_clean.parquet
    │   └── excluded_genomes_report.csv
    │
    ├── v2/
    │   ├── genomes_all/
    │   ├── genomes_excluded/
    │   ├── metadata/
    │   │   └── assembly_data_report.jsonl
    │   ├── sample_metadata_clean.parquet
    │   └── excluded_genomes_report.csv
    │
    └── v3/
        ├── genomes_all/
        ├── genomes_excluded/
        ├── metadata/
        │   └── assembly_data_report.jsonl
        ├── sample_metadata_clean.parquet
        └── excluded_genomes_report.csv

A lógica vigente é a de que para cada nova consulta ao NCBI em busca dos dados criaremos uma versão de dataset. Assim, por consulta recorrente e programada (script de schedule) o banco de dados no cluster será atualizado de acordo, compondo uma nova versão. Isso é importante para usar o mesmo *genome core* quando gerarmos os dados SNPS.

## **1.1. Imports**

In [1]:
import json
import shutil
import zipfile
import pandas as pd
import subprocess

from pathlib import Path



from clade_clustering.paths import RAW_DIR, MANIFESTS_DIR, PROCESSED_DIR
from clade_clustering.metadata import normalize_metadata_cols

## **1.2 Configuração do dataset**

Cada coleta ao NCBI constitui uma versão independente do dataset bruto.
A versão identifica um snapshot dos genomas e metadados disponíveis no
momento da aquisição.

In [2]:
DATA_VERSION = "v4"

SNAPSHOT_DIR = RAW_DIR / DATA_VERSION

GENOMES_ALL_DIR = SNAPSHOT_DIR / "genomes_all"
GENOMES_EXCLUDED_DIR = SNAPSHOT_DIR / "genomes_excluded"
METADATA_DIR = SNAPSHOT_DIR / "metadata"

NCBI_DOWNLOAD_DIR = SNAPSHOT_DIR / "ncbi_download"
NCBI_ZIP_PATH = SNAPSHOT_DIR / "cauris_all.zip"

METADATA_PATH = METADATA_DIR / "assembly_data_report.jsonl"


PROCESSED_VERSION_DIR = PROCESSED_DIR / DATA_VERSION

SAMPLE_METADATA_CLEAN_PATH = PROCESSED_VERSION_DIR / "sample_metadata_clean.parquet"
EXCLUDED_GENOMES_REPORT_PATH = PROCESSED_VERSION_DIR / "excluded_genomes_report.csv"

MANIFEST_DIR = MANIFESTS_DIR / DATA_VERSION

APPROVED_GENOMES_MANIFEST_PATH = MANIFEST_DIR / "approved_genomes.csv"



### **1.2.1 Estado do snapshot**

Rotina de proteção contra sobrescrever os arquivos silenciosamente.

In [3]:
snapshot_has_files = (
    SNAPSHOT_DIR.exists()
    and any(path.is_file() for path in SNAPSHOT_DIR.rglob("*"))
)

snapshot_has_raw_data = (
    METADATA_PATH.exists()
    and GENOMES_ALL_DIR.exists()
    and any(GENOMES_ALL_DIR.glob("*.fna"))
)

if not snapshot_has_files:
    DOWNLOAD_REQUIRED = True
    print(
        f"Snapshot {DATA_VERSION!r} ainda não existe. "
        "O dataset será baixado."
    )

elif snapshot_has_raw_data:
    DOWNLOAD_REQUIRED = False
    print(
        f"Snapshot {DATA_VERSION!r} já contém os dados brutos. "
        "O download será ignorado."
    )

else:
    raise RuntimeError(
        f"O snapshot {DATA_VERSION!r} existe, mas parece incompleto:\n"
        f"{SNAPSHOT_DIR}\n"
        "Revise o conteúdo antes de continuar."
    )

Snapshot 'v4' já contém os dados brutos. O download será ignorado.


### **1.2.2 Construção dos caminhos**

In [4]:
for path in [
    GENOMES_ALL_DIR,
    GENOMES_EXCLUDED_DIR,
    METADATA_DIR,
    NCBI_DOWNLOAD_DIR,
    MANIFEST_DIR,
    PROCESSED_VERSION_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

## **1.3 Download arquivos `.fna` e metadados**

Nesta seção utilizaremos o **NCBI Datasets CLI** para compor os dados `.fna` com as sequências completas dos genomas da *C. auris* e os respectivos metadados. Em suma, os seguintes arquivos relevantes serão baixados:


- `assembly_data_report.jsonl`: **arquivo com os metadados biológicos e técnicos de cada genoma. O formato JSONL estabelece que cada linha do arquivo é um JSON descrevendo cada genoma, ou seja, 1 linha = 1 genoma.**

- `dataset_catalog.json`: **É um inventário resumido sobre todos os arquivos que foram baixados.**

- `Arquivos FASTA (.fna)`: **Arquivos com as sequências completas dos genomas.**

### **1.3.1 Verificar existência do NCBI Datasets CLI**

In [5]:
DATASETS_BIN = shutil.which("datasets")

if DATASETS_BIN is None:
    raise RuntimeError(
        "NCBI Datasets CLI não encontrado no PATH."
    )

print(f"NCBI Datasets CLI encontrado em: {DATASETS_BIN}")

NCBI Datasets CLI encontrado em: /home/lucas/.local/bin/datasets


### **1.3.2 Baixando arquivo .zip com os genomas**

In [6]:
if DOWNLOAD_REQUIRED:
    cmd = [
        DATASETS_BIN,
        "download",
        "genome",
        "taxon",
        "Candidozyma auris",
        "--include",
        "genome",
        "--filename",
        str(NCBI_ZIP_PATH),
    ]

    print(f"Baixando snapshot {DATA_VERSION!r} do NCBI...")
    subprocess.run(cmd, check=True)

    if not NCBI_ZIP_PATH.exists():
        raise RuntimeError(
            f"O download terminou, mas o arquivo esperado não foi encontrado:\n"
            f"{NCBI_ZIP_PATH}"
        )

    print(f"Download concluído: {NCBI_ZIP_PATH}")

else:
    print(
        f"Snapshot {DATA_VERSION!r} já possui os dados brutos. "
        "Download ignorado."
    )

Snapshot 'v4' já possui os dados brutos. Download ignorado.


### **1.3.3 Validação do arquivo .zip baixado**

In [7]:
if not NCBI_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP do NCBI não encontrado:\n{NCBI_ZIP_PATH}"
    )

if not zipfile.is_zipfile(NCBI_ZIP_PATH):
    raise RuntimeError(
        f"O arquivo baixado não é um ZIP válido:\n{NCBI_ZIP_PATH}"
    )

size_mb = NCBI_ZIP_PATH.stat().st_size / (1024 ** 2)

print(f"ZIP válido: {NCBI_ZIP_PATH}")
print(f"Tamanho: {size_mb:.1f} MB")

ZIP válido: /home/lucas/workspace/projects/clade-clustering/data/raw/v4/cauris_all.zip
Tamanho: 4635.2 MB


In [8]:
with zipfile.ZipFile(NCBI_ZIP_PATH, "r") as zf:
    members = zf.namelist()

print(f"Número de entradas no ZIP: {len(members)}")

for member in members[:20]:
    print(member)

Número de entradas no ZIP: 1251
README.md
ncbi_dataset/data/assembly_data_report.jsonl
ncbi_dataset/data/GCA_001049995.1/GCA_001049995.1_VPCI_479_P_13_genomic.fna
ncbi_dataset/data/GCA_001189475.1/GCA_001189475.1_ASM118947v1_genomic.fna
ncbi_dataset/data/GCA_002775015.1/GCA_002775015.1_Cand_auris_B11221_V1_genomic.fna
ncbi_dataset/data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna
ncbi_dataset/data/GCA_003014415.1/GCA_003014415.1_Cand_auris_B11243_genomic.fna
ncbi_dataset/data/GCA_004287075.1/GCA_004287075.1_RCPF-1821_genomic.fna
ncbi_dataset/data/GCA_005234155.1/GCA_005234155.1_ASM523415v1_genomic.fna
ncbi_dataset/data/GCA_007168705.1/GCA_007168705.1_ASM716870v1_genomic.fna
ncbi_dataset/data/GCA_008275145.1/GCA_008275145.1_ASM827514v1_genomic.fna
ncbi_dataset/data/GCA_008729165.1/GCA_008729165.1_ASM872916v1_genomic.fna
ncbi_dataset/data/GCA_008729175.1/GCA_008729175.1_ASM872917v1_genomic.fna
ncbi_dataset/data/GCA_008729195.1/GCA_008729195.1_ASM872919v1_genomic.fna
ncbi_datas

In [9]:
if NCBI_DOWNLOAD_DIR.exists() and any(NCBI_DOWNLOAD_DIR.iterdir()):
    raise RuntimeError(
        f"O diretório de extração não está vazio:\n"
        f"{NCBI_DOWNLOAD_DIR}"
    )

with zipfile.ZipFile(NCBI_ZIP_PATH, "r") as zf:
    zf.extractall(NCBI_DOWNLOAD_DIR)

print(f"Pacote extraído em: {NCBI_DOWNLOAD_DIR}")

RuntimeError: O diretório de extração não está vazio:
/home/lucas/workspace/projects/clade-clustering/data/raw/v4/ncbi_download

In [ ]:
extracted_files = [
    path
    for path in NCBI_DOWNLOAD_DIR.rglob("*")
    if path.is_file()
]

print(f"Arquivos extraídos: {len(extracted_files)}")

Arquivos extraídos: 1251


### **1.3.4 Organizando os arquivos desempacotados do .zip**

- **Caminho explícito da raíz interna do pacote NCBI**

In [ ]:
NCBI_DATA_DIR = NCBI_DOWNLOAD_DIR / "ncbi_dataset" / "data"

ASSEMBLY_REPORT_SOURCE = (
    NCBI_DATA_DIR / "assembly_data_report.jsonl"
)

DATASET_CATALOG_SOURCE = (
    NCBI_DATA_DIR / "dataset_catalog.json"
)

METADATA_PATH = (
    METADATA_DIR / "assembly_data_report.jsonl"
)

DATASET_CATALOG_PATH = (
    METADATA_DIR / "dataset_catalog.json"
)

- **Inspeção do número de genomas baixados e se o `assembly_data_report.jsonl` existe**

In [ ]:
fna_files = sorted(NCBI_DATA_DIR.rglob("*_genomic.fna"))

print(f"Genomas encontrados: {len(fna_files)}")
print(f"Metadata de assembly encontrado: {ASSEMBLY_REPORT_SOURCE.exists()}")
print(f"Metadata de inventário/catálogo encontrado: {DATASET_CATALOG_SOURCE.exists()}")

Genomas encontrados: 0
Metadata de assembly encontrado: False
Metadata de inventário/catálogo encontrado: False


- **Checa se há pastas com os arquivos FASTA (.fna) de mesmo nome**

In [ ]:
fna_names = [path.name for path in fna_files]

if len(fna_names) != len(set(fna_names)):
    raise RuntimeError(
        "Há nomes de arquivos FASTA duplicados no pacote NCBI. "
        "Não é seguro achatar a estrutura."
    )

In [ ]:
fna_files

[]

- **Checar se há arquivos FASTA em .../genomes_all e metadados em .../metadata**

In [ ]:
existing_genomes = list(GENOMES_ALL_DIR.glob("*.fna"))

if existing_genomes:
    raise RuntimeError(
        f"{GENOMES_ALL_DIR} já contém "
        f"{len(existing_genomes)} arquivos FASTA."
    )

if not ASSEMBLY_REPORT_SOURCE.exists():
    raise FileNotFoundError(
        f"dataset_catalog.json não encontrado em:\n"
        f"{ASSEMBLY_REPORT_SOURCE}"
    )

if METADATA_PATH.exists():
    raise RuntimeError(
        f"O metadata de destino já existe:\n{METADATA_PATH}"
    )

if not DATASET_CATALOG_SOURCE.exists():
    raise FileNotFoundError(
        f"dataset_catalog.json não encontrado em:\n"
        f"{DATASET_CATALOG_SOURCE}"
    )

if DATASET_CATALOG_PATH.exists():
    raise RuntimeError(
        f"O catálogo de destino já existe:\n"
        f"{DATASET_CATALOG_PATH}"
    )

RuntimeError: /home/lucas/workspace/projects/clade-clustering/data/raw/v4/genomes_all já contém 1247 arquivos FASTA.

- **Checar o número total de genomas que serão movidos**

In [ ]:
fna_files = sorted(NCBI_DATA_DIR.rglob("*_genomic.fna"))
n_genomes_expected = len(fna_files)

print(f"Genomas encontrados: {n_genomes_expected}")

Genomas encontrados: 0


- **Movendo os arquivos fasta**

In [ ]:
for source_path in fna_files:
    destination_path = GENOMES_ALL_DIR / source_path.name
    shutil.move(str(source_path), str(destination_path))

shutil.move(
    str(ASSEMBLY_REPORT_SOURCE),
    str(METADATA_PATH),
)

shutil.move(
    str(DATASET_CATALOG_SOURCE),
    str(DATASET_CATALOG_PATH),
)

In [ ]:
organized_genomes = sorted(GENOMES_ALL_DIR.glob("*.fna"))

print(f"Genomas organizados: {len(organized_genomes)}")
print(f"Metadata: {METADATA_PATH}")
print(f"Metadata existe: {METADATA_PATH.exists()}")
print(f"Dataset catalog: {DATASET_CATALOG_PATH}")
print(f"Dataset catalog existe: {DATASET_CATALOG_PATH.exists()}")

if len(organized_genomes) != n_genomes_expected:
    raise RuntimeError(
        f"Número de FASTAs inconsistente: "
        f"esperados {n_genomes_expected}, "
        f"organizados {len(organized_genomes)}."
    )

if not METADATA_PATH.exists():
    raise RuntimeError(
        "O arquivo de metadata não foi movido corretamente."
    )

if not DATASET_CATALOG_PATH.exists():
    raise RuntimeError(
        "O dataset_catalog.json não foi movido corretamente."
    )

## **1.4 Pré-processamento dos metadados**

### **1.4.1. Leitura dos metadados brutos**

In [10]:
# Ler JSONL de metadados
# Cada linha é um JSON com metadados de um genoma
records = []

with open(METADATA_PATH, 'r') as f:
    for line in f: 
        records.append(json.loads(line)) # Lista de JSON's

# Converte o JSON aninhado em um df flat 
df_raw = pd.json_normalize(records)

print("Dimensão do metadata bruto", df_raw.shape)

Dimensão do metadata bruto (1247, 120)


### **1.4.2 Normalização e seleção**

Os nomes dos metadados vêm no formato `PascalCase` e `camelCase`:

In [11]:
df_raw.columns.to_list()[:5]

['currentAccession',
 'accession',
 'sourceDatabase',
 'assemblyInfo.assemblyLevel',
 'assemblyInfo.assemblyName']

Devemos portanto fazer a normalização para o formato `snakecase`:

In [12]:
df_norm = normalize_metadata_cols(df_raw)
df_norm.columns.to_list()[:5]

['current_accession',
 'accession',
 'source_database',
 'assembly_info.assembly_level',
 'assembly_info.assembly_name']

Selecionando conjuntos de metadados:

In [13]:
# Identidade da amostra
meta_sample_identity = [
    'accession', 'organism.organism_name', 
    'assembly_info.assembly_name', 
    'assembly_info.biosample.strain'
]

# Interpretação biológica/epidemiológica
meta_bio = [
    'assembly_info.biosample.lat_lon',
    'assembly_info.biosample.geo_loc_name',
    'assembly_info.biosample.collection_date',
    'assembly_info.biosample.isolation_source',
    'assembly_info.biosample.host'
]

# Qualidade técnica
meta_quality_control = [
    'assembly_info.sequencing_tech',
    'assembly_stats.total_sequence_length',
    'assembly_stats.contig_n50',         
    'assembly_stats.number_of_contigs', 
    'assembly_info.assembly_level',      
]

# Curadoria/status do assembly
meta_assembly_status = [
    'assembly_info.assembly_status',
    'assembly_info.atypical.is_atypical',
    'assembly_info.atypical.warnings',
    'assembly_info.genome_notes',
]

- **Proteção quanto uma eventual mudança de schema do NCBI**

In [14]:
meta_cols = meta_sample_identity + meta_assembly_status + meta_bio + meta_quality_control 


missing_meta_cols = [
    col for col in meta_cols
    if col not in df_norm.columns
]

if missing_meta_cols:
    raise RuntimeError(
        "Colunas esperadas ausentes no metadata do NCBI:\n"
        + "\n".join(missing_meta_cols)
    )

### **1.4.3 Construção das variáveis derivadas**

In [15]:
df_meta_raw = df_norm[meta_cols].copy()

# Renomear as colunas relevantes
df_meta_raw = df_meta_raw.rename(columns={
    # Identidade da amostra
    "accession":                                "accession",
    "organism.organism_name":                   "organism",
    "assembly_info.assembly_name":              "assembly_name",
    "assembly_info.biosample.strain":           "strain",
    # Curadoria/status do assembly
    'assembly_info.assembly_status':            "assembly_status",
    'assembly_info.atypical.is_atypical':       "assembly_is_atypical",
    'assembly_info.atypical.warnings':          "assembly_warnings",
    'assembly_info.genome_notes':               "assembly_notes",
    # Interpretação biológica/epidemiológica
    "assembly_info.biosample.geo_loc_name":     "country_raw",
    "assembly_info.biosample.lat_lon":          "coordinates",   
    "assembly_info.biosample.collection_date":  "collection_date_raw",
    "assembly_info.biosample.isolation_source": "isolation_source",
    "assembly_info.biosample.host":             "host",
    # Qualidade técnica
    "assembly_info.sequencing_tech":            "sequencing_tech",
    "assembly_stats.total_sequence_length":     "total_sequence_length",
    "assembly_stats.contig_n50":                "contig_n50",
    "assembly_stats.number_of_contigs":         "n_contigs",
    "assembly_info.assembly_level":             "assembly_level"
})

df_meta_raw.head()

,accession,organism,assembly_name,strain,assembly_status,assembly_is_atypical,assembly_warnings,assembly_notes,coordinates,country_raw,collection_date_raw,isolation_source,host,sequencing_tech,total_sequence_length,contig_n50,n_contigs,assembly_level
0,GCA_001049995.1,Candidozyma auris,VPCI 479/P/13,NaN,current,True,[partial],[partial],NaN,NaN,NaN,NaN,NaN,Illumina MiSeq,12308280,25974,825,Scaffold
1,GCA_001189475.1,Candidozyma auris,ASM118947v1,6684,current,NaN,NaN,[superseded by newer assembly for species],12.95 N 77.64 E,India: Bengaluru,2013-07,urine,Homo sapiens,Illumina HiSeq; Illumina MiSeq,12498763,30547,759,Scaffold
2,GCA_002775015.1,Candidozyma auris,Cand_auris_B11221_V1,B11221,current,NaN,NaN,[superseded by newer assembly for species],not collected,South Africa,2012,blood,Homo sapiens,PacBio,12741178,1467420,23,Scaffold
3,GCA_003013715.2,Candidozyma auris,ASM301371v2,B11220,current,NaN,NaN,NaN,not collected,Japan,2009,auditory canal,Homo sapiens,Oxford_Nanopore,12249773,2336890,7,Complete Genome
4,GCA_003014415.1,Candidozyma auris,Cand_auris_B11243,B11243,current,NaN,NaN,NaN,not collected,Venezuela,2013,blood,Homo sapiens,Illumina,12354602,77742,282,Scaffold


In [16]:
# Limpar país e ano
df_meta_raw['country'] = (
    df_meta_raw['country_raw']
    .astype('string').str.lower()
    .str.extract(r'([^:]+)')[0]
    .str.strip()
)

df_meta_raw['year'] = (
    df_meta_raw['collection_date_raw']
    .astype('string').str.extract(r'(\d{4})')[0]
)

# Converter campos numéricos de qualidade
df_meta_raw['size_mb']   = pd.to_numeric(df_meta_raw['total_sequence_length'], errors='coerce') / 1e6
df_meta_raw['contig_n50'] = pd.to_numeric(df_meta_raw['contig_n50'], errors='coerce')
df_meta_raw['n_contigs']  = pd.to_numeric(df_meta_raw['n_contigs'],  errors='coerce')

df_meta = df_meta_raw[[
    "accession", "organism", "assembly_name", 
    "strain", "assembly_status", "assembly_is_atypical",
    "assembly_warnings", "assembly_notes", "country", "coordinates",
    "year", "isolation_source", "host", 
    "sequencing_tech", "total_sequence_length", "size_mb",
    "contig_n50", "n_contigs", "assembly_level"
]].copy()

df_meta.head()

,accession,organism,assembly_name,strain,assembly_status,assembly_is_atypical,assembly_warnings,assembly_notes,country,coordinates,year,isolation_source,host,sequencing_tech,total_sequence_length,size_mb,contig_n50,n_contigs,assembly_level
0,GCA_001049995.1,Candidozyma auris,VPCI 479/P/13,NaN,current,True,[partial],[partial],<NA>,NaN,<NA>,NaN,NaN,Illumina MiSeq,12308280,12.308280,25974,825,Scaffold
1,GCA_001189475.1,Candidozyma auris,ASM118947v1,6684,current,NaN,NaN,[superseded by newer assembly for species],india,12.95 N 77.64 E,2013,urine,Homo sapiens,Illumina HiSeq; Illumina MiSeq,12498763,12.498763,30547,759,Scaffold
2,GCA_002775015.1,Candidozyma auris,Cand_auris_B11221_V1,B11221,current,NaN,NaN,[superseded by newer assembly for species],south africa,not collected,2012,blood,Homo sapiens,PacBio,12741178,12.741178,1467420,23,Scaffold
3,GCA_003013715.2,Candidozyma auris,ASM301371v2,B11220,current,NaN,NaN,NaN,japan,not collected,2009,auditory canal,Homo sapiens,Oxford_Nanopore,12249773,12.249773,2336890,7,Complete Genome
4,GCA_003014415.1,Candidozyma auris,Cand_auris_B11243,B11243,current,NaN,NaN,NaN,venezuela,not collected,2013,blood,Homo sapiens,Illumina,12354602,12.354602,77742,282,Scaffold


### **1.4.4 Cruzando metadados com arquivos `.fna`**

In [17]:
# fna_files = sorted(GENOMES_ALL_DIR.glob("*.fna"))

# df_files = pd.DataFrame({
#     "accession": [path.name for path in fna_files],
#     "fna_path": [str(path) for path in fna_files],
# })

accepted_files = sorted(GENOMES_ALL_DIR.glob("*.fna"))
excluded_files = sorted(GENOMES_EXCLUDED_DIR.glob("*.fna"))

fna_files = accepted_files + excluded_files

df_files = pd.DataFrame({
    "accession": [path.name for path in fna_files],
    "fna_path": [str(path) for path in fna_files],
})

df_files["accession"] = (
    df_files["accession"]
    .str.extract(r"^([^_]+_[^_]+)", expand=False)
)

df_files.head()

,accession,fna_path
0,GCA_002759435.3,/home/lucas/workspace/projects/clade-clusterin...
1,GCA_002775015.1,/home/lucas/workspace/projects/clade-clusterin...
2,GCA_003013715.2,/home/lucas/workspace/projects/clade-clusterin...
3,GCA_003014415.1,/home/lucas/workspace/projects/clade-clusterin...
4,GCA_004287075.1,/home/lucas/workspace/projects/clade-clusterin...


- **Merge buscando explicitamente um metadata para um Fasta por accession**

In [18]:
df_meta_files = df_meta.merge(
    df_files, 
    on="accession",
    how="left",
    validate="one_to_one"
)

df_meta_files

,accession,organism,assembly_name,strain,assembly_status,assembly_is_atypical,assembly_warnings,assembly_notes,country,coordinates,year,isolation_source,host,sequencing_tech,total_sequence_length,size_mb,contig_n50,n_contigs,assembly_level,fna_path
0,GCA_001049995.1,Candidozyma auris,VPCI 479/P/13,NaN,current,True,[partial],[partial],<NA>,NaN,<NA>,NaN,NaN,Illumina MiSeq,12308280,12.308280,25974,825,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
1,GCA_001189475.1,Candidozyma auris,ASM118947v1,6684,current,NaN,NaN,[superseded by newer assembly for species],india,12.95 N 77.64 E,2013,urine,Homo sapiens,Illumina HiSeq; Illumina MiSeq,12498763,12.498763,30547,759,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
2,GCA_002775015.1,Candidozyma auris,Cand_auris_B11221_V1,B11221,current,NaN,NaN,[superseded by newer assembly for species],south africa,not collected,2012,blood,Homo sapiens,PacBio,12741178,12.741178,1467420,23,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
3,GCA_003013715.2,Candidozyma auris,ASM301371v2,B11220,current,NaN,NaN,NaN,japan,not collected,2009,auditory canal,Homo sapiens,Oxford_Nanopore,12249773,12.249773,2336890,7,Complete Genome,/home/lucas/workspace/projects/clade-clusterin...
4,GCA_003014415.1,Candidozyma auris,Cand_auris_B11243,B11243,current,NaN,NaN,NaN,venezuela,not collected,2013,blood,Homo sapiens,Illumina,12354602,12.354602,77742,282,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1242,GCA_902712265.3,Candidozyma auris,Cau_1907_a,NaN,current,NaN,NaN,NaN,hong kong,NaN,2019,swab,Homo sapiens,Illumina MiSeq,12607765,12.607765,41270,592,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
1243,GCA_902712275.3,Candidozyma auris,Cau_1905_a,NaN,current,NaN,NaN,NaN,hong kong,NaN,2019,swab,Homo sapiens,Illumina MiSeq,12950902,12.950902,9316,2215,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
1244,GCA_902712835.3,Candidozyma auris,Cau_1901_a,NaN,current,NaN,NaN,NaN,hong kong,NaN,2019,Endotracheal aspirate,Homo sapiens,Illumina MiSeq,13049589,13.049589,7444,2730,Scaffold,/home/lucas/workspace/projects/clade-clusterin...
1245,GCA_947395165.1,Candidozyma auris,16B25_hybrid,NaN,current,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,"""MinION, Illumina""",12350990,12.350990,176725,111,Contig,/home/lucas/workspace/projects/clade-clusterin...


In [19]:
fna_without_metadata = df_files[
    ~df_files["accession"].isin(df_meta["accession"])
]

print("Metadata sem FASTA:", df_meta_files["fna_path"].isna().sum())
print("FASTA sem metadata:", len(fna_without_metadata))

Metadata sem FASTA: 0
FASTA sem metadata: 0


In [20]:
if df_meta_files["fna_path"].isna().any():
    raise RuntimeError(
        "Há registros de metadata sem FASTA correspondente."
    )

if not df_meta_files["fna_path"].map(
    lambda path: Path(path).is_file()
).all():
    raise RuntimeError(
        "Há caminhos de FASTA inválidos no dataset."
    )

### **1.4.5. Seleção de genomas: filtragem por mb e assemblies problemáticos**

In [21]:
# Parâmetros
SIZE_MB_MIN = 11

# ------------------ Máscaras de exclusão ------------------ #

mask_size_bad = df_meta_files["size_mb"] < SIZE_MB_MIN

mask_partial_assembly = (
    df_meta_files["assembly_notes"].astype(str).str.contains("partial", case=False, na=False)
    | df_meta_files["assembly_warnings"].astype(str).str.contains("partial", case=False, na=False)
)

mask_is_atypical = df_meta_files["assembly_is_atypical"] == True

mask_exclude = mask_size_bad | mask_partial_assembly | mask_is_atypical

n_before = len(df_meta_files)

# Diagnóstico do motivo da exclusão
df_meta_files["exclude_size_bad"] = mask_size_bad
df_meta_files["exclude_partial"] = mask_partial_assembly
df_meta_files["exclude_atypical"] = mask_is_atypical


def exclusion_reason(row):
    reasons = []

    if row["exclude_size_bad"]:
        reasons.append("size_mb_below_threshold")

    if row["exclude_partial"]:
        reasons.append("partial_assembly")

    if row["exclude_atypical"]:
        reasons.append("atypical_assembly")

    return "; ".join(reasons)


df_meta_files["exclusion_reason"] = df_meta_files.apply(exclusion_reason, axis=1)

# Aplicação da máscara
df_excluded = df_meta_files[mask_exclude].copy()
df_meta_files = df_meta_files[~mask_exclude].copy().reset_index(drop=True)


# ------------------ Mover genomas excluídos ------------------ #

new_paths = []

for _, row in df_excluded.iterrows():
    if pd.isna(row["fna_path"]):
        raise RuntimeError(
            f"{row['accession']}: fna_path ausente para genoma excluído."
        )

    src = Path(row["fna_path"]).resolve()
    dst = (GENOMES_EXCLUDED_DIR / src.name).resolve()

    # Já está corretamente em genomes_excluded/
    if src == dst and src.exists():
        print(f"[SKIP] já estava em genomes_excluded/: {dst.name}")
        new_paths.append(str(dst))
        continue

    # Primeira execução: mover de genomes_all/ para genomes_excluded/
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))
        print(
            f"[MOVE] excluído: {src.name} | "
            f"motivo: {row['exclusion_reason']}"
        )
        new_paths.append(str(dst))
        continue

    # Estado inconsistente: existem duas cópias
    if src.exists() and dst.exists():
        raise RuntimeError(
            f"FASTA duplicado em dois diretórios:\n"
            f"{src}\n{dst}"
        )

    # Arquivo já foi movido, mas fna_path ainda apontava para a origem
    if dst.exists():
        print(f"[SKIP] já estava em genomes_excluded/: {dst.name}")
        new_paths.append(str(dst))
        continue

    raise FileNotFoundError(
        f"FASTA não encontrado para {row['accession']}:\n"
        f"origem esperada: {src}\n"
        f"destino esperado: {dst}"
    )

df_excluded["excluded_fna_path"] = new_paths


# ------------------ Salvar relatório ------------------ #

report_cols = [
    "accession",
    "size_mb",
    "assembly_notes",
    "assembly_warnings",
    "assembly_is_atypical",
    "exclusion_reason",
    "fna_path",
    "excluded_fna_path",
]

df_excluded[report_cols].to_csv(
    EXCLUDED_GENOMES_REPORT_PATH,
    index=False
)

print(f"Genomas brutos          : {n_before}")
print(f"Excluídos por QC inicial: {len(df_excluded)}")
print(f"Restantes para análise  : {len(df_meta_files)}")
print(f"FASTA excluídos em      : {GENOMES_EXCLUDED_DIR}")
print(f"Relatório salvo em      : {EXCLUDED_GENOMES_REPORT_PATH}")

[SKIP] já estava em genomes_excluded/: GCA_001049995.1_VPCI_479_P_13_genomic.fna
[SKIP] já estava em genomes_excluded/: GCA_046252445.1_ASM4625244v1_genomic.fna
[SKIP] já estava em genomes_excluded/: GCA_049307545.1_ASM4930754v1_genomic.fna
Genomas brutos          : 1247
Excluídos por QC inicial: 3
Restantes para análise  : 1244
FASTA excluídos em      : /home/lucas/workspace/projects/clade-clustering/data/raw/v4/genomes_excluded
Relatório salvo em      : /home/lucas/workspace/projects/clade-clustering/data/processed/v4/excluded_genomes_report.csv


### **1.4.6. QC refinado: faixa de size_mb e fragmentação do assembly**

Aplicado diretamente sobre o `df_meta_files` da seção 1.4.4 (independe da 1.4.5):

- `size_mb` fora do intervalo [11, 13] Mb;
- `n_contigs` > 500 (assembly muito fragmentado).

Os genomas excluídos aqui são movidos para `genomes_excluded/`.

In [22]:
# # Parâmetros
# SIZE_MB_MIN = 11
# SIZE_MB_MAX = 13
# N_CONTIGS_MAX = 500

# # Esta célula espera o df_meta_files recém-gerado pelo merge da 1.4.4
# if "exclude_n_contigs_high" in df_meta_files.columns:
#     raise RuntimeError(
#         "QC refinado já aplicado neste df_meta_files. "
#         "Re-execute a partir da seção 1.4.4 antes desta."
#     )

# # ------------------ Máscaras de exclusão ------------------ #

# mask_size_out_of_range = ~df_meta_files["size_mb"].between(SIZE_MB_MIN, SIZE_MB_MAX)

# mask_n_contigs_high = df_meta_files["n_contigs"] > N_CONTIGS_MAX

# mask_exclude_refined = mask_size_out_of_range | mask_n_contigs_high

# n_before_refined = len(df_meta_files)

# # Diagnóstico do motivo da exclusão
# df_meta_files["exclude_size_out_of_range"] = mask_size_out_of_range
# df_meta_files["exclude_n_contigs_high"] = mask_n_contigs_high


# def refined_exclusion_reason(row):
#     reasons = []

#     if row["exclude_size_out_of_range"]:
#         reasons.append("size_mb_out_of_range")

#     if row["exclude_n_contigs_high"]:
#         reasons.append("n_contigs_above_threshold")

#     return "; ".join(reasons)


# df_meta_files["exclusion_reason"] = df_meta_files.apply(refined_exclusion_reason, axis=1)

# # Aplicação da máscara
# df_excluded_refined = df_meta_files[mask_exclude_refined].copy()
# df_meta_files = df_meta_files[~mask_exclude_refined].copy().reset_index(drop=True)


# # ------------------ Mover genomas excluídos ------------------ #

# new_paths = []

# for _, row in df_excluded_refined.iterrows():
#     if pd.isna(row["fna_path"]):
#         raise RuntimeError(
#             f"{row['accession']}: fna_path ausente para genoma excluído."
#         )

#     src = Path(row["fna_path"]).resolve()
#     dst = (GENOMES_EXCLUDED_DIR / src.name).resolve()

#     # Já está corretamente em genomes_excluded/
#     if src == dst and src.exists():
#         print(f"[SKIP] já estava em genomes_excluded/: {dst.name}")
#         new_paths.append(str(dst))
#         continue

#     # Primeira execução: mover de genomes_all/ para genomes_excluded/
#     if src.exists() and not dst.exists():
#         shutil.move(str(src), str(dst))
#         print(
#             f"[MOVE] excluído: {src.name} | "
#             f"motivo: {row['exclusion_reason']}"
#         )
#         new_paths.append(str(dst))
#         continue

#     # Estado inconsistente: existem duas cópias
#     if src.exists() and dst.exists():
#         raise RuntimeError(
#             f"FASTA duplicado em dois diretórios:\n"
#             f"{src}\n{dst}"
#         )

#     # Arquivo já foi movido, mas fna_path ainda apontava para a origem
#     if dst.exists():
#         print(f"[SKIP] já estava em genomes_excluded/: {dst.name}")
#         new_paths.append(str(dst))
#         continue

#     raise FileNotFoundError(
#         f"FASTA não encontrado para {row['accession']}:\n"
#         f"origem esperada: {src}\n"
#         f"destino esperado: {dst}"
#     )

# df_excluded_refined["excluded_fna_path"] = new_paths


# # ------------------ Salvar relatório ------------------ #

# report_cols_refined = [
#     "accession",
#     "size_mb",
#     "n_contigs",
#     "assembly_notes",
#     "assembly_warnings",
#     "assembly_is_atypical",
#     "exclusion_reason",
#     "fna_path",
#     "excluded_fna_path",
# ]

# # df_excluded_refined[report_cols_refined].to_csv(
# #     EXCLUDED_GENOMES_REPORT_PATH,
# #     index=False
# # )

# print(f"Genomas antes do QC refinado  : {n_before_refined}")
# print(f"  size_mb fora de [{SIZE_MB_MIN}, {SIZE_MB_MAX}]   : {mask_size_out_of_range.sum()}")
# print(f"  n_contigs > {N_CONTIGS_MAX}             : {mask_n_contigs_high.sum()}")
# print(f"Excluídos por QC refinado     : {len(df_excluded_refined)}")
# print(f"Restantes para análise        : {len(df_meta_files)}")

In [23]:
# Salvar metadados limpos
df_meta_files.to_parquet(SAMPLE_METADATA_CLEAN_PATH, index=False)

In [24]:
print(SAMPLE_METADATA_CLEAN_PATH.exists())

df_check = pd.read_parquet(SAMPLE_METADATA_CLEAN_PATH)
print(df_check.shape)

True
(1244, 24)


## **1.5 Criação do manifesto de genomas aprovados**

In [25]:
df_approved = pd.read_parquet(SAMPLE_METADATA_CLEAN_PATH)

manifest = df_approved[["accession", "fna_path"]].copy()

manifest["fasta_filename"] = manifest["fna_path"].map(
    lambda path: Path(path).name
)

manifest = (
    manifest[["accession", "fasta_filename"]]
    .sort_values("accession")
    .reset_index(drop=True)
)

manifest.to_csv(
    APPROVED_GENOMES_MANIFEST_PATH,
    index=False,
)

print(f"Manifesto salvo em: {APPROVED_GENOMES_MANIFEST_PATH}")
print(f"Genomas aprovados: {len(manifest)}")

Manifesto salvo em: /home/lucas/workspace/projects/clade-clustering/manifests/v4/approved_genomes.csv
Genomas aprovados: 1244
